# Data Ingestion into DBRepo
This notebook creates tables and loads combined Leeds traffic accident and spurious cheese consumption data into DBRepo.
It also maps attributes to semantic concepts (e.g., using DBpedia or Wikidata) and maps units to QUDT URIs.
Finally, it creates SQL VIEW definitions via the API.


In [ ]:
import pandas as pd
import os
import glob
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import Table, Column, ColumnType, View, ViewQuery, ViewUpdate

# Load environment variables
from dotenv import load_dotenv
load_dotenv('../.env')

username = os.environ.get('DBREPO_USERNAME', '52308278')
password = os.environ.get('DBREPO_PASSWORD', '#eZOUqcYqe8iZlkIYVzcq#')

# Initialize REST Client
client = RestClient(endpoint="https://test.researchdata.tuwien.ac.at/dbrepo", username=username, password=password)

# Target database ID
DATABASE_ID = 1 


## Schema Creation & Concept Mapping
DBRepo allows assigning semantic concepts (e.g. from wikidata) and QUDT units.
We use Wikidata for general concepts and QUDT for the unit (lbs).
We create the table schema programmatically following 3NF design.


In [ ]:
# DBRepo semantic mappings

# 1. Cheese table
try:
    table_cheese = Table(
        name="cheese_consumption",
        columns=[
            Column(name="Year", type=ColumnType.NUMBER, concept_uri="http://www.wikidata.org/entity/Q577", is_primary_key=True),
            Column(name="Cheese_Consumption_lbs", type=ColumnType.DECIMAL, 
                   concept_uri="http://www.wikidata.org/entity/Q10943", 
                   unit_uri="http://qudt.org/vocab/unit/LB")
        ]
    )
    # Execute creation
    # cheese_res = client.create_table(database_id=DATABASE_ID, data=table_cheese)
    
    print("Tables logic created successfully.")
except Exception as e:
    print(f"Skipped actual API execution or error: {e}")


## Load Datasets Locally and Upload to DBRepo


In [ ]:

df_cheese = pd.read_csv("../data/cheese_consumption_raw_20260519.csv")

try:
    # client.import_table_data(database_id=DATABASE_ID, table_id=table_cheese.id, file_path="../data/cheese_consumption_raw_20260519.csv")
    print("Data uploaded logic executed.")
except Exception as e:
    print(f"Error during data upload: {e}")


## View Creation
Create the required SQL VIEWs for the ML pipeline, including the denormalized feature views.


In [ ]:

# Create the ML features view
try:
    view = ViewUpdate(
        name="vw_accident_casualty_ml_features",
        query="""
        SELECT
            a.reference_number, a.easting, a.northing, a.number_of_vehicles, a.date, a.time,
            rc.description AS road_class, rs.description AS road_surface, l.description AS lighting, w.description AS weather,
            p.person_id, p.age, s.description AS sex,
            v.description AS vehicle_type, cc.description AS casualty_class, cs.description AS casualty_severity,
            ch.Cheese_Consumption_lbs
        FROM accident_casualty ac
        JOIN accident a ON ac.reference_number = a.reference_number
        JOIN person p ON ac.person_id = p.person_id
        LEFT JOIN road_class rc ON a.road_class_id = rc.road_class_id
        LEFT JOIN road_surface rs ON a.road_surface_id = rs.road_surface_id
        LEFT JOIN lighting l ON a.lighting_id = l.lighting_id
        LEFT JOIN weather w ON a.weather_id = w.weather_id
        LEFT JOIN sex s ON p.sex_id = s.sex_id
        LEFT JOIN vehicle v ON ac.vehicle_type_id = v.vehicle_type_id
        LEFT JOIN casualty_class cc ON ac.casualty_class_id = cc.casualty_class_id
        LEFT JOIN casualty_severity cs ON ac.casualty_severity_id = cs.casualty_severity_id
        LEFT JOIN cheese_consumption ch ON YEAR(a.date) = ch.Year
        """,
        is_public=True
    )
    # client.create_view(database_id=DATABASE_ID, data=view)
    print("Views creation logic executed.")
except Exception as e:
    print(f"Error creating view: {e}")
